# Deep Learning Practical – 7 Experiments
**Jupyter Notebook source**

This notebook contains one separate implementation block for each question:
1. Boston Housing price estimation (feed-forward regression)
2. IMDB positive/negative review classification
3. Reuters newswire topic classification
4. CNN for MNIST
5. MNIST classification using a pretrained model (MobileNetV2 transfer learning)
6. Time-series forecasting using LSTM
7. Autoencoder

**Requirements:** Python 3.9+, TensorFlow/Keras, scikit-learn, pandas, NumPy, Matplotlib.

Run the setup cell once, then execute each experiment block from top to bottom. Dataset downloads require an internet connection. Training time depends on your computer/GPU.

In [ ]:
# BLOCK 0: Install/import libraries (run once)
# If needed, uncomment the next line in Jupyter/Colab:
# %pip install -q tensorflow scikit-learn pandas numpy matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, classification_report

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


---
## 1. Deep feed-forward neural network – Boston Housing price estimation

**Task:** Predict the median value of owner-occupied homes from numerical housing features. This is a regression problem.

The original Boston Housing dataset is not bundled with current Keras releases. The code loads the classic dataset from OpenML. If OpenML is unavailable, check your internet connection.

In [ ]:
# BLOCK 1: Boston Housing regression
from sklearn.datasets import fetch_openml

# Load the classic Boston Housing dataset from OpenML
boston = fetch_openml(name="boston", version=1, as_frame=True, parser="auto")
X = boston.data.apply(pd.to_numeric, errors="coerce")
y = pd.to_numeric(boston.target, errors="coerce")

data = pd.concat([X, y.rename("MEDV")], axis=1).dropna()
X = data.drop(columns=["MEDV"]).to_numpy(dtype="float32")
y = data["MEDV"].to_numpy(dtype="float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype("float32")
X_test = scaler.transform(X_test).astype("float32")

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)  # Linear output for price regression
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(
    X_train, y_train, epochs=100, batch_size=16,
    validation_split=0.2, verbose=0,
    callbacks=[keras.callbacks.EarlyStopping(
        patience=12, restore_best_weights=True
    )]
)

test_loss, test_mae = model.evaluate(X_test, y_test, verbose=0)
predictions = model.predict(X_test[:5], verbose=0).flatten()

print("Boston Housing – test results")
print("Test MSE:", round(float(test_loss), 3))
print("Test MAE (in $1000s):", round(float(test_mae), 3))
print("Actual prices:", np.round(y_test[:5], 2))
print("Predicted prices:", np.round(predictions, 2))

plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Boston Housing – training history")
plt.legend()
plt.show()


**Output:** The cell prints test MSE, test MAE, five actual house values and five predicted values, followed by a training/validation loss graph. Values can vary slightly with software versions and training.

---
## 2. Feed-forward neural network – IMDB movie review sentiment

**Task:** Classify each movie review as positive (1) or negative (0). Reviews are represented as sequences of integer word IDs and converted to fixed-size multi-hot vectors.

In [ ]:
# BLOCK 2: IMDB binary sentiment classification
from tensorflow.keras.datasets import imdb

NUM_WORDS = 10000
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=NUM_WORDS)

# Convert each review into a 10,000-dimensional multi-hot vector
def vectorize_reviews(sequences, dimension=NUM_WORDS):
    result = np.zeros((len(sequences), dimension), dtype="float32")
    for i, sequence in enumerate(sequences):
        result[i, sequence] = 1.0
    return result

x_train_vec = vectorize_reviews(x_train)
x_test_vec = vectorize_reviews(x_test)

model = keras.Sequential([
    layers.Input(shape=(NUM_WORDS,)),
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

history = model.fit(
    x_train_vec, np.asarray(y_train),
    epochs=10, batch_size=512, validation_split=0.2, verbose=1,
    callbacks=[keras.callbacks.EarlyStopping(
        patience=2, restore_best_weights=True
    )]
)

test_loss, test_acc = model.evaluate(x_test_vec, y_test, verbose=0)
probabilities = model.predict(x_test_vec[:5], verbose=0).flatten()

print("\nIMDB test loss:", round(float(test_loss), 4))
print("IMDB test accuracy:", round(float(test_acc), 4))
print("Predicted sentiment:", ["positive" if p >= 0.5 else "negative" for p in probabilities])
print("Actual labels:", ["positive" if v == 1 else "negative" for v in y_test[:5]])

plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("IMDB sentiment classification")
plt.legend()
plt.show()


**Output:** Training progress, test loss/accuracy, predicted sentiment for five reviews, actual labels and an accuracy graph.

---
## 3. Deep neural network – Reuters newswire topic classification

**Task:** Assign each newswire to one of 46 topics. This is a multi-class classification problem, so the output layer uses softmax and the loss is categorical cross-entropy.

In [ ]:
# BLOCK 3: Reuters multi-class news classification
from tensorflow.keras.datasets import reuters
from tensorflow.keras.utils import to_categorical

NUM_WORDS = 10000
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=NUM_WORDS)

def vectorize_sequences(sequences, dimension=NUM_WORDS):
    result = np.zeros((len(sequences), dimension), dtype="float32")
    for i, sequence in enumerate(sequences):
        result[i, sequence] = 1.0
    return result

x_train_vec = vectorize_sequences(x_train)
x_test_vec = vectorize_sequences(x_test)
num_classes = int(max(np.max(y_train), np.max(y_test)) + 1)

y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

model = keras.Sequential([
    layers.Input(shape=(NUM_WORDS,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(
    x_train_vec, y_train_cat,
    epochs=15, batch_size=512, validation_split=0.2, verbose=1,
    callbacks=[keras.callbacks.EarlyStopping(
        patience=2, restore_best_weights=True
    )]
)

test_loss, test_acc = model.evaluate(x_test_vec, y_test_cat, verbose=0)
predicted_classes = np.argmax(model.predict(x_test_vec[:10], verbose=0), axis=1)

print("\nNumber of topic classes:", num_classes)
print("Reuters test loss:", round(float(test_loss), 4))
print("Reuters test accuracy:", round(float(test_acc), 4))
print("Predicted topic IDs:", predicted_classes)
print("Actual topic IDs:   ", y_test[:10])

plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Categorical cross-entropy")
plt.title("Reuters topic classification")
plt.legend()
plt.show()


**Output:** Number of classes (46), test loss/accuracy, predicted and actual topic IDs, and a loss graph.

---
## 4. CNN – MNIST handwritten digit classification

**Task:** Classify 28×28 grayscale images into digits 0–9 using convolution, pooling and dense layers.

In [ ]:
# BLOCK 4: CNN on MNIST
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values and add the grayscale channel dimension
x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]

model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Conv2D(64, kernel_size=3, activation="relu"),
    layers.MaxPooling2D(pool_size=2),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

history = model.fit(
    x_train, y_train, epochs=5, batch_size=128,
    validation_split=0.1, verbose=1
)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
predictions = np.argmax(model.predict(x_test[:10], verbose=0), axis=1)

print("\nMNIST CNN test loss:", round(float(test_loss), 4))
print("MNIST CNN test accuracy:", round(float(test_acc), 4))
print("Predicted digits:", predictions)
print("Actual digits:   ", y_test[:10])

fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_test[i].squeeze(), cmap="gray")
    ax.set_title(f"Pred: {predictions[i]} / True: {y_test[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()


**Output:** Test accuracy/loss, predicted and actual digit labels, and sample handwritten digit images with predictions.

---
## 5. MNIST classification using a pretrained model (transfer learning)

**Task:** Reuse ImageNet-pretrained MobileNetV2 as a frozen feature extractor, replace its classification head with a 10-class MNIST head, and train the new head.

This is a demonstration of transfer learning. MobileNetV2 expects 3-channel images, so grayscale MNIST images are resized and repeated across three channels. Pretrained weights are downloaded by Keras.

In [ ]:
# BLOCK 5: Transfer learning with pretrained MobileNetV2
from tensorflow.keras.datasets import mnist
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Keep the data manageable for this transfer-learning demonstration.
# Increase these limits if you have sufficient RAM/GPU memory.
TRAIN_LIMIT = 12000
TEST_LIMIT = 2000
x_train = x_train[:TRAIN_LIMIT]
y_train = y_train[:TRAIN_LIMIT]
x_test = x_test[:TEST_LIMIT]
y_test = y_test[:TEST_LIMIT]

def prepare_mnist_for_mobilenet(images):
    images = tf.convert_to_tensor(images[..., np.newaxis], dtype=tf.float32)
    images = tf.image.resize(images, (96, 96))
    images = tf.image.grayscale_to_rgb(images)
    return preprocess_input(images)  # maps pixel values to [-1, 1]

x_train_rgb = prepare_mnist_for_mobilenet(x_train)
x_test_rgb = prepare_mnist_for_mobilenet(x_test)

base_model = MobileNetV2(
    weights="imagenet", include_top=False, input_shape=(96, 96, 3)
)
base_model.trainable = False  # freeze pretrained convolutional layers

inputs = keras.Input(shape=(96, 96, 3))
features = base_model(inputs, training=False)
features = layers.GlobalAveragePooling2D()(features)
features = layers.Dropout(0.2)(features)
outputs = layers.Dense(10, activation="softmax")(features)
model = keras.Model(inputs, outputs)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(
    x_train_rgb, y_train, epochs=3, batch_size=64,
    validation_split=0.1, verbose=1
)

test_loss, test_acc = model.evaluate(x_test_rgb, y_test, verbose=0)
predictions = np.argmax(model.predict(x_test_rgb[:10], verbose=0), axis=1)

print("\nPretrained MobileNetV2 – MNIST subset")
print("Training samples:", len(x_train_rgb), "| Test samples:", len(x_test_rgb))
print("Test loss:", round(float(test_loss), 4))
print("Test accuracy:", round(float(test_acc), 4))
print("Predicted digits:", predictions)
print("Actual digits:   ", y_test[:10])


**Output:** Training progress, test accuracy/loss on the selected MNIST subset, and predicted versus actual digit labels. This model is pretrained on ImageNet, not on MNIST; only its new classification head is trained here. Results may be lower than a CNN trained directly on MNIST.

---
## 6. Time-series forecasting using an LSTM

**Task:** Learn a sequence pattern from previous time steps and predict the next value. A synthetic sine wave is used so the experiment runs without an external dataset. Replace it with your own time-series values when available.

In [ ]:
# BLOCK 6: LSTM time-series forecasting
# Create a reproducible time series (sine wave + small noise)
time = np.arange(0, 1200, dtype="float32")
series = np.sin(0.03 * time) + 0.3 * np.sin(0.07 * time)
series += np.random.normal(0, 0.05, size=len(time)).astype("float32")

WINDOW = 30
split_index = int(len(series) * 0.8)

# Build windows without allowing training windows to use future test targets
def make_windows(values, window):
    X, y = [], []
    for i in range(len(values) - window):
        X.append(values[i:i + window])
        y.append(values[i + window])
    return np.array(X, dtype="float32"), np.array(y, dtype="float32")

X_all, y_all = make_windows(series, WINDOW)
train_end = split_index - WINDOW
X_train, y_train = X_all[:train_end], y_all[:train_end]
X_test, y_test = X_all[train_end:], y_all[train_end:]

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

model = keras.Sequential([
    layers.Input(shape=(WINDOW, 1)),
    layers.LSTM(32),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

history = model.fit(
    X_train, y_train, epochs=20, batch_size=32,
    validation_split=0.2, verbose=1,
    callbacks=[keras.callbacks.EarlyStopping(
        patience=4, restore_best_weights=True
    )]
)

predictions = model.predict(X_test, verbose=0).flatten()
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)

print("\nLSTM forecasting results")
print("Test RMSE:", round(float(rmse), 4))
print("Test MAE:", round(float(mae), 4))

plt.figure(figsize=(12, 4))
plt.plot(y_test[:200], label="Actual")
plt.plot(predictions[:200], label="Predicted")
plt.title("LSTM – actual vs predicted time series")
plt.xlabel("Test time step")
plt.ylabel("Value")
plt.legend()
plt.show()


**Output:** Test RMSE and MAE, plus a graph comparing actual and predicted values for the first 200 test time steps.

---
## 7. Autoencoder – MNIST image reconstruction

**Task:** Train an encoder to compress each image into a smaller latent representation and a decoder to reconstruct the original image. The model is trained without class labels.

In [ ]:
# BLOCK 7: Convolutional autoencoder on MNIST
from tensorflow.keras.datasets import mnist

(x_train, _), (x_test, _) = mnist.load_data()
x_train = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_test = (x_test.astype("float32") / 255.0)[..., np.newaxis]

# Encoder
encoder_input = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu", padding="same")(encoder_input)
x = layers.MaxPooling2D(2, padding="same")(x)
x = layers.Conv2D(16, 3, activation="relu", padding="same")(x)
latent = layers.MaxPooling2D(2, padding="same", name="compressed_representation")(x)
encoder = keras.Model(encoder_input, latent, name="encoder")

# Decoder
decoder_input = keras.Input(shape=(7, 7, 16))
x = layers.Conv2D(16, 3, activation="relu", padding="same")(decoder_input)
x = layers.UpSampling2D(2)(x)
x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
x = layers.UpSampling2D(2)(x)
decoder_output = layers.Conv2D(1, 3, activation="sigmoid", padding="same")(x)
decoder = keras.Model(decoder_input, decoder_output, name="decoder")

autoencoder_input = keras.Input(shape=(28, 28, 1))
encoded = encoder(autoencoder_input)
reconstructed = decoder(encoded)
autoencoder = keras.Model(autoencoder_input, reconstructed, name="autoencoder")

autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
autoencoder.fit(
    x_train, x_train, epochs=5, batch_size=128,
    validation_split=0.1, shuffle=True, verbose=1
)

test_loss = autoencoder.evaluate(x_test, x_test, verbose=0)
reconstructed_images = autoencoder.predict(x_test[:10], verbose=0)

print("\nAutoencoder test reconstruction loss:", round(float(test_loss), 5))
print("Original image shape:", x_test[0].shape)
print("Encoded representation shape:", encoder.predict(x_test[:1], verbose=0).shape)
print("Reconstructed image shape:", reconstructed_images[0].shape)

# Compare original and reconstructed images
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    axes[0, i].imshow(x_test[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(reconstructed_images[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("Original")
axes[1, 0].set_ylabel("Reconstructed")
plt.suptitle("MNIST Autoencoder reconstruction")
plt.tight_layout()
plt.show()


**Output:** Reconstruction loss, original/encoded/reconstructed tensor shapes, and a two-row image plot showing original digits above their reconstructions.

---
## Practical notes and expected outputs

- Neural-network scores are **not fixed**. They vary with random initialization, library versions, hardware and training duration. The notebook prints the actual metrics produced by your run.
- Boston Housing is a small historical dataset and has known ethical/statistical limitations; use it here as a classroom regression exercise, not for real property valuation.
- The MobileNetV2 experiment demonstrates transfer learning on a limited MNIST subset and is computationally heavier than the CNN experiment.
- The time-series experiment uses synthetic data. For real forecasting, replace `series` with a chronologically ordered numeric series and use a time-based train/test split.
- Save trained models with `model.save("model_name.keras")` if you need to reuse them.